# EDA de importaciones ARCA - Proyecto Medallion
**Bootcamp:** Fundamentos de Ingeniería de Datos
**Tema:** EDA (Exploratory Data Analysis) con SQL
**Dataset:** `bootcamp.bronze.arca_importaciones_bronze` (importaciones por ítem de declaración aduanera, ARCA - corte `202610`)

**Instrucciones**
1. Usar `%sql` al inicio de cada celda SQL
2. Documentar los hallazgos en comentarios
3. Subir las soluciones a GitHub

> **Importante:** un buen EDA es la base de un buen pipeline. Los hallazgos de acá definen las reglas de Silver.
> **Ajustá el nombre de la tabla Bronze** si el tuyo es distinto (está escrito como `bootcamp.bronze.arca_importaciones_bronze`).

# **Creación de tabla en bronze**
Exactamente como vienen los valores desde origen

In [0]:

%sql
--PRIMERA VERIFICACIÓN:
SELECT * FROM read_files(
  '/Volumes/bootcamp/landing/proyecto/impo_parte_0000.parquet',
  format => 'parquet',
  header => true
)
LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TABLE bootcamp.bronze.arca_importaciones_bronze AS
SELECT
  *,
  _metadata.file_path AS _archivo_origen,
  current_timestamp() AS _ingesta_ts
FROM read_files(
  '/Volumes/bootcamp/landing/proyecto/impo_parte_*.parquet',
  format => 'parquet'
);

### **⚠️ Hallazgo importante:**
Comparando el archivo original que era un .lst (archivo de texto en formato viejo) que pesaba más de 5gb, vs 72.84 MB en formato parquet y usando archivos delta. ![image_1794271460995.png](./image_1794271460995.png "image_1794271460995.png") 
Es mucho el ahorro de espacio con archivos .parquet

## Contexto del dataset
- **Fuente:** ARCA, "Información de comercio exterior" (archivo mensual `impo_AAAAMM.lst`, reporte de texto).
- https://www.arca.gob.ar/operadoresComercioExterior/informacionAgregada/informacion-agregada.asp
- **Grano esperado:** una fila por *destinación + ítem + concepto arancelario* (un mismo ítem puede repetirse con distintos conceptos).
- **Ingesta:** el `.lst` pesa ~5 GB por mes (líneas de 700 caracteres, ~153 de datos y el resto relleno). Databricks Free no admite esa carga, así que se redujo a Parquet en local (script propio, corte por posición) y se subió al Volume.
- **Regla de Bronze:** todo como texto, sin castear y sin filtrar. El tipado y la limpieza van en Silver.
- **Ojo:** el importador figura por nombre (sin CUIT) y viene cortado a 30 caracteres.

---
# PARTE 1: Exploración inicial
**Importante en bronze: dejar TODO tal cual te llega**

### Ejercicio 1.1: Verificar datos disponibles

¿Cuántos registros hay en total en la tabla Bronze? Tiene que coincidir con las filas leídas por el script de conversión (8.026.601).

*Pista: usá COUNT(\*)*

In [0]:
%sql
SELECT COUNT(*) AS filas
FROM bootcamp.bronze.arca_importaciones_bronze;

-- Hallazgos:
-- 

### Ejercicio 1.2: Explorar estructura

¿Qué columnas tiene la tabla? ¿Qué tipos de datos tienen? ¿Qué columnas de auditoría agregó Bronze?

*Pista: usá DESCRIBE TABLE*

In [0]:
%sql
DESCRIBE bootcamp.bronze.arca_importaciones_bronze;

-- Hallazgos:
-- 

### Ejercicio 1.3: Ver muestra de datos

Mostrá 10 filas con las columnas más importantes: `destinacion, num_item, periodo, nombre_importador, pos_ncm, pais_origen, unidad_medida, cantidad, divisa, fob_dolar, fob_total, concepto_arancel, monto_arancel`.

*Pista: SELECT con columnas específicas y LIMIT 10*

In [0]:
%sql
SELECT
  destinacion,
  num_item,
  periodo,
  nombre_importador,
  pos_ncm, --PRODUCTO CON CODIGO DEL MERCOSUR QUE SE IMPORTÓ (Posición NCM: Nomenclatura Común del Mercosu)
  pais_origen, --aparece como código. Averiguar si se puede conseguir por algún lado. (SÍ DE RESOLUCION 1507 - 2003). Creado una tabla auxiliar en bronze: ref_pais_aduana
  unidad_medida,
  cantidad,
  divisa,
  fob_dolar,
  fob_total,
  concepto_arancel,
  monto_arancel
FROM bootcamp.bronze.arca_importaciones_bronze
LIMIT 10;

-- Hallazgos:
-- 

In [0]:
%sql 
--QUIERO VER QUE TANTO PESAN LOS PAÍSES DE ORIGEN EN EL DATASET: 


CREATE OR REPLACE TEMPORARY VIEW pais_resumen AS
SELECT
  pais_origen,
  COUNT(*)                          AS filas,
  COUNT(DISTINCT destinacion)       AS despachos,
  COUNT(DISTINCT nombre_importador) AS importadores
FROM bootcamp.bronze.arca_importaciones_bronze
GROUP BY pais_origen;

SELECT
  *,
  ROUND(filas * 100.0 / SUM(filas) OVER (), 2) AS pct_filas
FROM pais_resumen
ORDER BY filas DESC;

In [0]:
%sql
SELECT *
FROM bootcamp.bronze.arca_importaciones_bronze
WHERE NOT (pais_origen RLIKE '^[0-9]{3}$')
LIMIT 20;

In [0]:
%sql
-- =====================================================================
-- Tabla de referencia: códigos de país de la Aduana argentina
-- Fuente: Anexo de países de la RG AFIP 1507 (2003), publicada en el Boletín Oficial.
-- OJO:
--   * Son códigos de la Aduana argentina, NO ISO ni los del SII de Chile.
--   * La RG 1507 (2003) es una norma abrogada: asumimos que el sistema de
--     códigos se mantiene, y lo validamos contra los datos
--     (ej.: 310 = China, 203 = Brasil, con capítulos NCM coherentes).
--   * No es un catálogo completo: los códigos que no están acá se tratan
--     como 'SIN IDENTIFICAR' en Silver (ver fin del script).
-- =====================================================================

CREATE OR REPLACE TABLE bootcamp.bronze.ref_pais_aduana (
  codigo STRING COMMENT 'Código numérico de 3 dígitos de la Aduana argentina (pais_origen / pais_procedencia)',
  pais   STRING COMMENT 'Nombre del país según el anexo de países de la RG AFIP 1507',
  fuente STRING COMMENT 'Documento de donde se tomó el código'
)
COMMENT 'Referencia de códigos de país de la Aduana argentina, tomada del anexo de la RG AFIP 1507 (2003). Cobertura parcial: hay códigos presentes en los datos sin identificar.';

INSERT INTO bootcamp.bronze.ref_pais_aduana VALUES
 -- ---------- América (serie 2xx) ----------
 ('202','Bolivia','Anexo RG 1507-2003'),
 ('203','Brasil','Anexo RG 1507-2003'),
 ('204','Canadá','Anexo RG 1507-2003'),
 ('205','Colombia','Anexo RG 1507-2003'),
 ('208','Chile','Anexo RG 1507-2003'),
 ('210','Ecuador','Anexo RG 1507-2003'),
 ('212','Estados Unidos','Anexo RG 1507-2003'),
 ('218','México','Anexo RG 1507-2003'),
 ('221','Paraguay','Anexo RG 1507-2003'),
 ('222','Perú','Anexo RG 1507-2003'),
 ('225','Uruguay','Anexo RG 1507-2003'),
 ('226','Venezuela','Anexo RG 1507-2003'),

 -- ---------- África (serie 1xx) ----------
 ('127','Marruecos','Anexo RG 1507-2003'),
 ('131','Nigeria','Anexo RG 1507-2003'),
 ('159','Sudáfrica','Anexo RG 1507-2003'),

 -- ---------- Asia y Medio Oriente (serie 3xx) ----------
 ('302','Arabia Saudita','Anexo RG 1507-2003'),
 ('303','Bahrein','Anexo RG 1507-2003'),
 ('308','Corea Democrática','Anexo RG 1507-2003'),
 ('309','Corea Republicana','Anexo RG 1507-2003'),
 ('310','China','Anexo RG 1507-2003'),
 ('312','Filipinas','Anexo RG 1507-2003'),
 ('313','Taiwán','Anexo RG 1507-2003'),
 ('315','India','Anexo RG 1507-2003'),
 ('316','Indonesia','Anexo RG 1507-2003'),
 ('319','Israel','Anexo RG 1507-2003'),
 ('320','Japón','Anexo RG 1507-2003'),
 ('321','Jordania','Anexo RG 1507-2003'),
 ('325','Líbano','Anexo RG 1507-2003'),
 ('326','Malasia','Anexo RG 1507-2003'),
 ('331','Emiratos Árabes Unidos','Anexo RG 1507-2003'),
 ('332','Pakistán','Anexo RG 1507-2003'),
 ('333','Singapur','Anexo RG 1507-2003'),
 ('334','Siria','Anexo RG 1507-2003'),
 ('335','Tailandia','Anexo RG 1507-2003'),
 ('337','Vietnam','Anexo RG 1507-2003'),
 ('341','Hong Kong','Anexo RG 1507-2003'),

 -- ---------- Europa (serie 4xx) ----------
 ('405','Austria','Anexo RG 1507-2003'),
 ('406','Bélgica','Anexo RG 1507-2003'),
 ('407','Bulgaria','Anexo RG 1507-2003'),
 ('409','Dinamarca','Anexo RG 1507-2003'),
 ('410','España','Anexo RG 1507-2003'),
 ('411','Finlandia','Anexo RG 1507-2003'),
 ('412','Francia','Anexo RG 1507-2003'),
 ('413','Grecia','Anexo RG 1507-2003'),
 ('414','Hungría','Anexo RG 1507-2003'),
 ('415','Irlanda','Anexo RG 1507-2003'),
 ('416','Islandia','Anexo RG 1507-2003'),
 ('417','Italia','Anexo RG 1507-2003'),
 ('420','Malta','Anexo RG 1507-2003'),
 ('422','Noruega','Anexo RG 1507-2003'),
 ('423','Países Bajos','Anexo RG 1507-2003'),
 ('424','Polonia','Anexo RG 1507-2003'),
 ('425','Portugal','Anexo RG 1507-2003'),
 ('426','Reino Unido','Anexo RG 1507-2003'),
 ('427','Rumania','Anexo RG 1507-2003'),
 ('429','Suecia','Anexo RG 1507-2003'),
 ('430','Suiza','Anexo RG 1507-2003'),
 ('436','Turquía','Anexo RG 1507-2003'),
 ('438','Alemania','Anexo RG 1507-2003'),
 ('442','Lituania','Anexo RG 1507-2003'),
 ('444','Rusia','Anexo RG 1507-2003'),
 ('445','Ucrania','Anexo RG 1507-2003'),
 ('449','Eslovenia','Anexo RG 1507-2003'),
 ('451','República Checa','Anexo RG 1507-2003'),

 -- ---------- Oceanía (serie 5xx) ----------
 ('501','Australia','Anexo RG 1507-2003');

-- ---------------------------------------------------------------------
-- Códigos que aparecen en los datos y NO pudimos identificar (sin fuente):
--   345, 306, 448, 141, 206, 280
-- No se inventan: en Silver quedan como 'SIN IDENTIFICAR'.
-- ---------------------------------------------------------------------

---
# PARTE 2: Análisis de valores nulos y vacíos

> **Ojo:** en Bronze todo es texto y el script le sacó los espacios a cada campo. Un campo sin dato llega como cadena vacía (`''`), no como `NULL`. Por eso se cuenta `IS NULL OR TRIM(col) = ''`.

### Ejercicio 2.1: Contar nulos o vacíos por columna

¿Cuántos valores nulos o vacíos hay en cada columna?

*Pista: usá COUNT_IF(col IS NULL OR TRIM(col) = '') para cada columna.*

In [0]:
%sql
SELECT
  COUNT_IF(aduana IS NULL OR TRIM(aduana) = '') AS aduana,
  COUNT_IF(destinacion IS NULL OR TRIM(destinacion) = '') AS destinacion,
  COUNT_IF(num_item IS NULL OR TRIM(num_item) = '') AS num_item,
  COUNT_IF(periodo IS NULL OR TRIM(periodo) = '') AS periodo,
  COUNT_IF(nombre_importador IS NULL OR TRIM(nombre_importador) = '') AS nombre_importador,
  COUNT_IF(medio_transporte IS NULL OR TRIM(medio_transporte) = '') AS medio_transporte,
  COUNT_IF(unidad_medida IS NULL OR TRIM(unidad_medida) = '') AS unidad_medida,
  COUNT_IF(cantidad IS NULL OR TRIM(cantidad) = '') AS cantidad,
  COUNT_IF(fob_dolar IS NULL OR TRIM(fob_dolar) = '') AS fob_dolar,
  COUNT_IF(fob_total IS NULL OR TRIM(fob_total) = '') AS fob_total,
  COUNT_IF(divisa IS NULL OR TRIM(divisa) = '') AS divisa,
  COUNT_IF(pais_origen IS NULL OR TRIM(pais_origen) = '') AS pais_origen,
  COUNT_IF(pais_procedencia IS NULL OR TRIM(pais_procedencia) = '') AS pais_procedencia,
  COUNT_IF(pos_ncm IS NULL OR TRIM(pos_ncm) = '') AS pos_ncm,
  COUNT_IF(concepto_arancel IS NULL OR TRIM(concepto_arancel) = '') AS concepto_arancel,
  COUNT_IF(monto_arancel IS NULL OR TRIM(monto_arancel) = '') AS monto_arancel
FROM bootcamp.bronze.arca_importaciones_bronze;

-- Hallazgos:
-- 

### Ejercicio 2.2: Porcentaje de nulos o vacíos

¿Qué porcentaje de registros tiene valores nulos o vacíos en cada columna? Mostrá solo las que tengan algún caso, en formato largo.

*Pista: ROUND(COUNT_IF(...) * 100.0 / COUNT(\*), 2) y UNPIVOT para pasar columnas a filas.*

In [0]:
%sql
WITH pct_nulos AS (
  SELECT
    ROUND(COUNT_IF(aduana IS NULL OR TRIM(aduana) = '') * 100.0 / COUNT(*), 2) AS aduana,
    ROUND(COUNT_IF(destinacion IS NULL OR TRIM(destinacion) = '') * 100.0 / COUNT(*), 2) AS destinacion,
    ROUND(COUNT_IF(num_item IS NULL OR TRIM(num_item) = '') * 100.0 / COUNT(*), 2) AS num_item,
    ROUND(COUNT_IF(periodo IS NULL OR TRIM(periodo) = '') * 100.0 / COUNT(*), 2) AS periodo,
    ROUND(COUNT_IF(nombre_importador IS NULL OR TRIM(nombre_importador) = '') * 100.0 / COUNT(*), 2) AS nombre_importador,
    ROUND(COUNT_IF(medio_transporte IS NULL OR TRIM(medio_transporte) = '') * 100.0 / COUNT(*), 2) AS medio_transporte,
    ROUND(COUNT_IF(unidad_medida IS NULL OR TRIM(unidad_medida) = '') * 100.0 / COUNT(*), 2) AS unidad_medida,
    ROUND(COUNT_IF(cantidad IS NULL OR TRIM(cantidad) = '') * 100.0 / COUNT(*), 2) AS cantidad,
    ROUND(COUNT_IF(fob_dolar IS NULL OR TRIM(fob_dolar) = '') * 100.0 / COUNT(*), 2) AS fob_dolar,
    ROUND(COUNT_IF(fob_total IS NULL OR TRIM(fob_total) = '') * 100.0 / COUNT(*), 2) AS fob_total,
    ROUND(COUNT_IF(divisa IS NULL OR TRIM(divisa) = '') * 100.0 / COUNT(*), 2) AS divisa,
    ROUND(COUNT_IF(pais_origen IS NULL OR TRIM(pais_origen) = '') * 100.0 / COUNT(*), 2) AS pais_origen,
    ROUND(COUNT_IF(pais_procedencia IS NULL OR TRIM(pais_procedencia) = '') * 100.0 / COUNT(*), 2) AS pais_procedencia,
    ROUND(COUNT_IF(pos_ncm IS NULL OR TRIM(pos_ncm) = '') * 100.0 / COUNT(*), 2) AS pos_ncm,
    ROUND(COUNT_IF(concepto_arancel IS NULL OR TRIM(concepto_arancel) = '') * 100.0 / COUNT(*), 2) AS concepto_arancel,
    ROUND(COUNT_IF(monto_arancel IS NULL OR TRIM(monto_arancel) = '') * 100.0 / COUNT(*), 2) AS monto_arancel
  FROM bootcamp.bronze.arca_importaciones_bronze
)
SELECT columna, porcentaje_nulos
FROM pct_nulos
UNPIVOT (
  porcentaje_nulos FOR columna IN (
    aduana, destinacion, num_item, periodo, nombre_importador, medio_transporte, unidad_medida, cantidad, fob_dolar, fob_total, divisa, pais_origen, pais_procedencia, pos_ncm, concepto_arancel, monto_arancel
  )
)
WHERE porcentaje_nulos > 0
ORDER BY porcentaje_nulos DESC;

-- Hallazgos:
-- 

### Ejercicio 2.3: Identificar columnas críticas

¿Qué columnas tienen más del 5% de valores nulos o vacíos? ¿Cuáles son críticas para el análisis (importador, NCM, país, valor FOB, cantidad)?

*Pista: reutilizá el ejercicio anterior y cambiá el umbral del WHERE.*

In [0]:
%sql
WITH pct_nulos AS (
  SELECT
    ROUND(COUNT_IF(aduana IS NULL OR TRIM(aduana) = '') * 100.0 / COUNT(*), 2) AS aduana,
    ROUND(COUNT_IF(destinacion IS NULL OR TRIM(destinacion) = '') * 100.0 / COUNT(*), 2) AS destinacion,
    ROUND(COUNT_IF(num_item IS NULL OR TRIM(num_item) = '') * 100.0 / COUNT(*), 2) AS num_item,
    ROUND(COUNT_IF(periodo IS NULL OR TRIM(periodo) = '') * 100.0 / COUNT(*), 2) AS periodo,
    ROUND(COUNT_IF(nombre_importador IS NULL OR TRIM(nombre_importador) = '') * 100.0 / COUNT(*), 2) AS nombre_importador,
    ROUND(COUNT_IF(medio_transporte IS NULL OR TRIM(medio_transporte) = '') * 100.0 / COUNT(*), 2) AS medio_transporte,
    ROUND(COUNT_IF(unidad_medida IS NULL OR TRIM(unidad_medida) = '') * 100.0 / COUNT(*), 2) AS unidad_medida,
    ROUND(COUNT_IF(cantidad IS NULL OR TRIM(cantidad) = '') * 100.0 / COUNT(*), 2) AS cantidad,
    ROUND(COUNT_IF(fob_dolar IS NULL OR TRIM(fob_dolar) = '') * 100.0 / COUNT(*), 2) AS fob_dolar,
    ROUND(COUNT_IF(fob_total IS NULL OR TRIM(fob_total) = '') * 100.0 / COUNT(*), 2) AS fob_total,
    ROUND(COUNT_IF(divisa IS NULL OR TRIM(divisa) = '') * 100.0 / COUNT(*), 2) AS divisa,
    ROUND(COUNT_IF(pais_origen IS NULL OR TRIM(pais_origen) = '') * 100.0 / COUNT(*), 2) AS pais_origen,
    ROUND(COUNT_IF(pais_procedencia IS NULL OR TRIM(pais_procedencia) = '') * 100.0 / COUNT(*), 2) AS pais_procedencia,
    ROUND(COUNT_IF(pos_ncm IS NULL OR TRIM(pos_ncm) = '') * 100.0 / COUNT(*), 2) AS pos_ncm,
    ROUND(COUNT_IF(concepto_arancel IS NULL OR TRIM(concepto_arancel) = '') * 100.0 / COUNT(*), 2) AS concepto_arancel,
    ROUND(COUNT_IF(monto_arancel IS NULL OR TRIM(monto_arancel) = '') * 100.0 / COUNT(*), 2) AS monto_arancel
  FROM bootcamp.bronze.arca_importaciones_bronze
)
SELECT columna, porcentaje_nulos
FROM pct_nulos
UNPIVOT (
  porcentaje_nulos FOR columna IN (
    aduana, destinacion, num_item, periodo, nombre_importador, medio_transporte, unidad_medida, cantidad, fob_dolar, fob_total, divisa, pais_origen, pais_procedencia, pos_ncm, concepto_arancel, monto_arancel
  )
)
WHERE porcentaje_nulos > 5
ORDER BY porcentaje_nulos DESC;

-- Hallazgos:
-- 

---
# PARTE 3: Cardinalidad y distribución

### Ejercicio 3.1: Distribución de divisa

¿Qué valores hay en `divisa`? ¿Cuántos registros hay de cada una? ¿Hay códigos distintos de `DOL`?

*Pista: GROUP BY divisa con conteo y porcentaje. La tabla de códigos está en el PDF de ARCA "Cómo descargar y leer los archivos de información de comercio exterior".*

In [0]:
%sql
SELECT
  divisa,
  COUNT(*) AS total,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS porcentaje
FROM bootcamp.bronze.arca_importaciones_bronze
GROUP BY divisa
ORDER BY total DESC;

-- Hallazgos:
-- 

In [0]:
%sql 
--PARA CAPA SILVER: ESTO VIENE DE LA RESOLUCIÓN RG 1886

CASE
  WHEN divisa = 'DOL' THEN 'Dólar estadounidense'
  WHEN divisa = '060' THEN 'Euro'
  WHEN divisa = 'PES' THEN 'Peso argentino'
  WHEN divisa = '012' THEN 'Real'
  WHEN divisa = '019' THEN 'Yen'
  WHEN divisa = '021' THEN 'Libra esterlina'
  WHEN divisa = '009' THEN 'Franco suizo'
  WHEN divisa = '016' THEN 'Corona sueca'
  WHEN divisa = '018' THEN 'Dólar canadiense'
  WHEN divisa = '034' THEN 'Rand sudafricano'
  WHEN divisa = '026' THEN 'Dólar australiano'
  WHEN divisa = '033' THEN 'Peso chileno'
  WHEN divisa = '014' THEN 'Corona danesa'
  WHEN divisa = '011' THEN 'Peso uruguayo'
  WHEN divisa = '032' THEN 'Peso colombiano'
  WHEN divisa = '030' THEN 'Shekel'
  -- WHEN divisa = '061' THEN '<Yuan o Zloty polaco: definir con la consulta de validación>'
  -- WHEN divisa = '063' THEN '<definir con la consulta de validación>'
  WHEN divisa IS NULL OR TRIM(divisa) = '' THEN 'SIN DATO'
  ELSE 'SIN IDENTIFICAR'
END AS divisa_limpio

### Ejercicio 3.2: Distribución por capítulo NCM

¿Cuántas filas hay por capítulo NCM (los 2 primeros dígitos de `pos_ncm`)? ¿Qué porcentaje cae en los capítulos 28 a 40 (química)?

*Pista: SUBSTRING(pos_ncm, 1, 2) y GROUP BY.*

In [0]:
%sql
SELECT
  SUBSTRING(pos_ncm, 1, 2) AS capitulo_ncm,
  COUNT(*) AS total,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS porcentaje
FROM bootcamp.bronze.arca_importaciones_bronze
GROUP BY SUBSTRING(pos_ncm, 1, 2)
ORDER BY capitulo_ncm;

-- Hallazgos:
-- 

### Ejercicio 3.3: Distribución de medio de transporte

¿Qué medios de transporte hay y qué peso tiene cada uno?

*Pista: GROUP BY medio_transporte ORDER BY total DESC*

In [0]:
%sql
SELECT
  medio_transporte,
  COUNT(*) AS total,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS porcentaje
FROM bootcamp.bronze.arca_importaciones_bronze
GROUP BY medio_transporte
ORDER BY total DESC;

-- Hallazgos:
-- 

### Ejercicio 3.4: Top países de origen

¿Cuáles son los 15 países de origen con más filas? Mostrá código, cantidad y porcentaje del total.

*Pista: GROUP BY pais_origen ORDER BY total DESC LIMIT 15. Los códigos de país necesitan una tabla de equivalencias para Silver.*

In [0]:
%sql
SELECT
  pais_origen,
  COUNT(*) AS total,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS porcentaje
FROM bootcamp.bronze.arca_importaciones_bronze
GROUP BY pais_origen
ORDER BY total DESC
LIMIT 15;

-- Hallazgos:
-- 

### Ejercicio 3.5: Distribución de concepto arancelario

¿Qué conceptos arancelarios hay (`concepto_arancel`)? Un mismo ítem puede repetirse con distintos conceptos, ¿cuáles son los más frecuentes?

*Pista: GROUP BY concepto_arancel ORDER BY total DESC*

In [0]:
%sql
SELECT
  concepto_arancel,
  COUNT(*) AS total,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS porcentaje
FROM bootcamp.bronze.arca_importaciones_bronze
GROUP BY concepto_arancel
ORDER BY total DESC
LIMIT 15;

-- Hallazgos:
-- 

### Ejercicio 3.6: Distribución de unidad de medida

¿Qué unidades de medida hay? ¿Se pueden comparar cantidades entre unidades distintas?

*Pista: GROUP BY unidad_medida ORDER BY total DESC*

In [0]:
%sql
SELECT
  unidad_medida,
  COUNT(*) AS total,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS porcentaje
FROM bootcamp.bronze.arca_importaciones_bronze
GROUP BY unidad_medida
ORDER BY total DESC
LIMIT 15;

-- Hallazgos:
-- 

---
# PARTE 4: Estadísticas descriptivas de variables numéricas

> **Nota sobre formato de datos:** en Bronze las columnas numéricas son strings. Creamos vistas temporales que corrigen los tipos.
> La vista temporal vive hasta que se apaga el cómputo. Si se apaga, EJECUTAR DE VUELTA.
> **Cuidado con TRY_CAST:** si el dato importa y no se puede convertir, devuelve NULL. Por eso en 5.1 se cuenta cuántos NULL aparecen.

In [0]:
%sql
-- Vista 1: columnas casteadas, una fila por línea del archivo
CREATE OR REPLACE TEMPORARY VIEW impo_clean AS
SELECT
  aduana,
  destinacion,
  TRY_CAST(num_item AS INT)         AS num_item,
  periodo,
  nombre_importador,
  medio_transporte,
  unidad_medida,
  TRY_CAST(cantidad AS DOUBLE)      AS cantidad,
  TRY_CAST(fob_dolar AS DOUBLE)     AS fob_dolar,
  TRY_CAST(fob_total AS DOUBLE)     AS fob_total,
  divisa,
  pais_origen,
  pais_procedencia,
  pos_ncm,
  SUBSTRING(pos_ncm, 1, 2)          AS capitulo_ncm,
  concepto_arancel,
  TRY_CAST(monto_arancel AS DOUBLE) AS monto_arancel
FROM bootcamp.bronze.arca_importaciones_bronze;

SELECT * FROM impo_clean LIMIT 10;

In [0]:
%sql
-- Vista 2: una fila por destinación + ítem (evita contar dos veces un ítem que se repite por concepto arancelario)
-- Solo divisa = 'DOL' hasta confirmar qué significan los demás códigos
CREATE OR REPLACE TEMPORARY VIEW impo_items AS
SELECT
  destinacion,
  num_item,
  MAX(nombre_importador) AS nombre_importador,
  MAX(pos_ncm)           AS pos_ncm,
  MAX(capitulo_ncm)      AS capitulo_ncm,
  MAX(pais_origen)       AS pais_origen,
  MAX(fob_dolar)         AS fob_dolar,
  SUM(monto_arancel)     AS arancel_total
FROM impo_clean
WHERE divisa = 'DOL'
GROUP BY destinacion, num_item;

SELECT * FROM impo_items LIMIT 10;

### Ejercicio 4.1: Estadísticas de FOB por divisa

**Pregunta:** para cada divisa, calculá: cantidad de registros, mínimo, máximo, promedio, mediana, percentil 25 y percentil 75 de `fob_dolar`. Solo valores válidos (mayores a 0).

*Pista: MIN(), MAX(), AVG(), PERCENTILE_APPROX() con GROUP BY divisa. Se usa la versión aproximada porque son millones de filas.*

In [0]:
%sql
SELECT
  divisa,
  COUNT(*)                                       AS cantidad,
  MIN(fob_dolar)                                 AS minimo,
  MAX(fob_dolar)                                 AS maximo,
  ROUND(AVG(fob_dolar), 2)                       AS promedio,
  ROUND(PERCENTILE_APPROX(fob_dolar, 0.5), 2)    AS mediana,
  ROUND(PERCENTILE_APPROX(fob_dolar, 0.25), 2)   AS p25,
  ROUND(PERCENTILE_APPROX(fob_dolar, 0.75), 2)   AS p75
FROM impo_clean
WHERE fob_dolar > 0
GROUP BY divisa
ORDER BY cantidad DESC;

-- Hallazgos:
-- 

### Ejercicio 4.2: Estadísticas de cantidad por unidad de medida

**Pregunta:** calculá mínimo, máximo, promedio y mediana de `cantidad` separado por `unidad_medida`. No tiene sentido comparar kilos con unidades.

*Pista: GROUP BY unidad_medida, solo cantidad > 0.*

In [0]:
%sql
SELECT
  unidad_medida,
  COUNT(*)                                      AS cantidad_filas,
  MIN(cantidad)                                 AS minimo,
  MAX(cantidad)                                 AS maximo,
  ROUND(AVG(cantidad), 2)                       AS promedio,
  ROUND(PERCENTILE_APPROX(cantidad, 0.5), 2)    AS mediana
FROM impo_clean
WHERE cantidad > 0
GROUP BY unidad_medida
ORDER BY cantidad_filas DESC;

-- Hallazgos:
-- 

### Ejercicio 4.3: Análisis del monto de arancel por concepto

**Pregunta:** para cada `concepto_arancel`, mostrá cantidad de filas, mínimo, máximo, promedio y cuántas tienen monto 0.

*Pista: COUNT_IF(monto_arancel = 0). Ayuda a entender qué conceptos son tributos reales y cuáles son otros cargos.*

In [0]:
%sql
SELECT
  concepto_arancel,
  COUNT(*)                                AS filas,
  MIN(monto_arancel)                      AS minimo,
  MAX(monto_arancel)                      AS maximo,
  ROUND(AVG(monto_arancel), 2)            AS promedio,
  COUNT_IF(monto_arancel = 0)             AS con_monto_cero
FROM impo_clean
GROUP BY concepto_arancel
ORDER BY filas DESC
LIMIT 20;

-- Hallazgos:
-- 

### Ejercicio 4.4: Detección de valores placeholder

**Objetivo:** identificar valores que se repiten demasiado y pueden ser relleno (0, 1, 999, 999999).

**Pregunta:** ¿cuáles son los 10 valores más frecuentes de `cantidad` y de `fob_dolar`?

*Pista: GROUP BY valor ORDER BY total DESC LIMIT 10. Los paréntesis permiten usar ORDER BY y LIMIT dentro de un UNION ALL.*

In [0]:
%sql
(SELECT 'cantidad' AS variable, CAST(cantidad AS STRING) AS valor, COUNT(*) AS total
 FROM impo_clean GROUP BY cantidad ORDER BY total DESC LIMIT 10)
UNION ALL
(SELECT 'fob_dolar' AS variable, CAST(fob_dolar AS STRING) AS valor, COUNT(*) AS total
 FROM impo_clean GROUP BY fob_dolar ORDER BY total DESC LIMIT 10);

-- Hallazgos:
-- 

---
# PARTE 5: Detección de problemas de calidad

### Ejercicio 5.1: Reporte de calidad usando CTEs

Creá un reporte que muestre: total de registros, FOB inválido (NULL o <= 0), cantidad inválida (NULL o <= 0), NCM con formato raro, importador vacío, divisa distinta de `DOL`, país de origen vacío, monto de arancel NULL o negativo, y el porcentaje de cada problema sobre el total.

*Pista: CTEs para totales y para métricas, y después un CROSS JOIN. El formato NCM esperado es `NNNN.NN.NN`.*

In [0]:
%sql
WITH total AS (
  SELECT COUNT(*) AS gran_total
  FROM impo_clean
),
metricas AS (
  SELECT
    COUNT_IF(fob_dolar IS NULL OR fob_dolar <= 0)                                  AS fob_inv,
    COUNT_IF(cantidad IS NULL OR cantidad <= 0)                                    AS cantidad_inv,
    COUNT_IF(NOT (pos_ncm RLIKE '^[0-9]{4}[.][0-9]{2}[.][0-9]{2}$'))               AS ncm_inv,
    COUNT_IF(nombre_importador IS NULL OR TRIM(nombre_importador) = '')            AS importador_inv,
    COUNT_IF(divisa <> 'DOL')                                                      AS divisa_no_dol,
    COUNT_IF(pais_origen IS NULL OR TRIM(pais_origen) = '')                        AS pais_inv,
    COUNT_IF(monto_arancel IS NULL OR monto_arancel < 0)                           AS arancel_inv
  FROM impo_clean
)
SELECT
  total.gran_total,
  metricas.*,
  ROUND(fob_inv * 100.0 / total.gran_total, 2)        AS porc_fob_inv,
  ROUND(cantidad_inv * 100.0 / total.gran_total, 2)   AS porc_cantidad_inv,
  ROUND(ncm_inv * 100.0 / total.gran_total, 4)        AS porc_ncm_inv,
  ROUND(importador_inv * 100.0 / total.gran_total, 4) AS porc_importador_inv,
  ROUND(divisa_no_dol * 100.0 / total.gran_total, 2)  AS porc_divisa_no_dol,
  ROUND(pais_inv * 100.0 / total.gran_total, 4)       AS porc_pais_inv,
  ROUND(arancel_inv * 100.0 / total.gran_total, 4)    AS porc_arancel_inv
FROM total
CROSS JOIN metricas;

-- Hallazgos:
-- 

### Ejercicio 5.2: Detectar duplicados

¿Hay filas duplicadas? Agrupá por `destinacion + num_item + concepto_arancel` (el grano esperado) y mostrá: cantidad de grupos duplicados, total de registros duplicados y cuántos registros "extra" hay.

*Pista: GROUP BY ... HAVING COUNT(\*) > 1, y después sumá.*

In [0]:
%sql
WITH grupos AS (
  SELECT destinacion, num_item, concepto_arancel, COUNT(*) AS veces
  FROM impo_clean
  GROUP BY destinacion, num_item, concepto_arancel
  HAVING COUNT(*) > 1
)
SELECT
  COUNT(*)       AS grupos_duplicados,
  SUM(veces)     AS registros_duplicados,
  SUM(veces - 1) AS registros_extra
FROM grupos;

-- Hallazgos:
-- 

### Ejercicio 5.3: Ver ejemplos de duplicados

Mostrá 10 ejemplos de grupos duplicados, ordenados por cantidad de apariciones descendente.

*Pista: el mismo GROUP BY, ORDER BY veces DESC LIMIT 10.*

In [0]:
%sql
SELECT destinacion, num_item, concepto_arancel, COUNT(*) AS veces
FROM impo_clean
GROUP BY destinacion, num_item, concepto_arancel
HAVING COUNT(*) > 1
ORDER BY veces DESC
LIMIT 10;

-- Hallazgos:
-- 

### Ejercicio 5.4: ¿`fob_total` es el total de la destinación?

**Hipótesis (a verificar):** `fob_total` se repite igual en todos los ítems de una misma destinación, o sea es un total del despacho y no del ítem. Si es así, sumar `fob_total` fila por fila duplica montos.

**Pregunta:** solo en `DOL`, tomá un valor por ítem y calculá por destinación la suma de `fob_dolar` y la cantidad de valores distintos de `fob_total`. ¿Cuántas destinaciones tienen más de un valor distinto? ¿En cuántas la suma por ítem difiere de `fob_total` en más de 1 USD?

*Pista: CTE para un valor por ítem, CTE por destinación, y un resumen con SUM(CASE WHEN ...).*

In [0]:
%sql
WITH items AS (
  SELECT
    destinacion,
    num_item,
    MAX(fob_dolar) AS fob_item,
    MAX(fob_total) AS fob_tot
  FROM impo_clean
  WHERE divisa = 'DOL'
  GROUP BY destinacion, num_item
),
por_dest AS (
  SELECT
    destinacion,
    SUM(fob_item)           AS suma_items,
    COUNT(DISTINCT fob_tot) AS n_fob_total,
    MAX(fob_tot)            AS fob_total
  FROM items
  GROUP BY destinacion
)
SELECT
  COUNT(*)                                                           AS destinaciones,
  SUM(CASE WHEN n_fob_total > 1 THEN 1 ELSE 0 END)                   AS con_fob_total_variable,
  SUM(CASE WHEN ABS(suma_items - fob_total) > 1 THEN 1 ELSE 0 END)   AS no_cierran
FROM por_dest;

-- Hallazgos:
-- 

### Ejercicio 5.5: Detectar outliers en FOB

Calculá los percentiles 1 y 99 de `fob_dolar` por divisa. Después contá cuántas filas quedan fuera de ese rango y mostrá las 20 más grandes.

*Pista: CTE con PERCENTILE_APPROX por divisa, y un JOIN. Los valores muy altos pueden ser errores de carga, y también operaciones legítimas muy grandes: no se borran sin revisar.*

In [0]:
%sql
WITH limites AS (
  SELECT
    divisa,
    PERCENTILE_APPROX(fob_dolar, 0.01) AS percentil_1,
    PERCENTILE_APPROX(fob_dolar, 0.99) AS percentil_99
  FROM impo_clean
  WHERE fob_dolar > 0
  GROUP BY divisa
)
SELECT
  p.divisa,
  COUNT_IF(p.fob_dolar < l.percentil_1) AS bajo_p1,
  COUNT_IF(p.fob_dolar > l.percentil_99) AS sobre_p99,
  MAX(l.percentil_1)  AS percentil_1,
  MAX(l.percentil_99) AS percentil_99
FROM impo_clean p
JOIN limites l ON p.divisa = l.divisa
WHERE p.fob_dolar > 0
GROUP BY p.divisa;

-- Hallazgos:
-- RECORDAR: un percentil N es el VALOR por debajo del cual cae el N% de los datos.
-- Outliers = fob_dolar < percentil_1 OR fob_dolar > percentil_99

In [0]:
%sql
-- Las 20 filas con mayor fob_dolar (para revisarlas a mano)
SELECT destinacion, num_item, nombre_importador, pos_ncm, pais_origen, divisa, cantidad, unidad_medida, fob_dolar
FROM impo_clean
WHERE fob_dolar > 0
ORDER BY fob_dolar DESC
LIMIT 20;

### Ejercicio 5.6: Calidad del nombre del importador

El nombre viene cortado a 30 caracteres y sin CUIT. ¿Cuántos nombres distintos hay? ¿Cuántos miden exactamente 30 caracteres (posiblemente cortados)? ¿Cuántos nombres distintos quedan si se normaliza (mayúsculas, sin signos ni espacios)?

*Pista: LENGTH(), UPPER() y REGEXP_REPLACE(). Esto define si hace falta una dimensión de importadores en Silver.*

In [0]:
%sql
SELECT
  COUNT(DISTINCT nombre_importador)                                                 AS nombres_distintos,
  COUNT(DISTINCT CASE WHEN LENGTH(nombre_importador) = 30 THEN nombre_importador END) AS nombres_de_30_caracteres,
  COUNT(DISTINCT REGEXP_REPLACE(UPPER(nombre_importador), '[^A-Z0-9]', ''))         AS nombres_normalizados
FROM impo_clean;

-- Hallazgos:
-- 

---
# PARTE 6: Análisis avanzado con Window Functions

> De acá en adelante se usa `impo_items` (una fila por destinación + ítem, solo `DOL`) para no contar dos veces un ítem repetido por concepto arancelario.

### Ejercicio 6.1: Ranking de capítulos NCM por valor FOB

Mostrá, para cada capítulo NCM: FOB total en USD, cantidad de ítems y el ranking (1 = más importado).

*Pista: calculá el total por capítulo y usá ROW_NUMBER() OVER (ORDER BY SUM(fob_dolar) DESC).*

In [0]:
%sql
SELECT
  capitulo_ncm,
  ROUND(SUM(fob_dolar), 2) AS fob_total_usd,
  COUNT(*)                 AS items,
  ROW_NUMBER() OVER (ORDER BY SUM(fob_dolar) DESC) AS ranking
FROM impo_items
WHERE fob_dolar > 0
GROUP BY capitulo_ncm
ORDER BY ranking;

-- Hallazgos:
-- 

### Ejercicio 6.2: Comparar cada capítulo con el promedio general

Para cada capítulo mostrá: FOB promedio por ítem, el FOB promedio general, la diferencia y el porcentaje de diferencia.

*Pista: AVG(fob_dolar) OVER() da el promedio general, o una CTE con CROSS JOIN.*

In [0]:
%sql
WITH general AS (
  SELECT AVG(fob_dolar) AS promedio_gral
  FROM impo_items
  WHERE fob_dolar > 0
),
por_capitulo AS (
  SELECT
    capitulo_ncm,
    AVG(fob_dolar) AS fob_promedio_capitulo,
    COUNT(*)       AS items
  FROM impo_items
  WHERE fob_dolar > 0
  GROUP BY capitulo_ncm
)
SELECT
  c.capitulo_ncm,
  c.items,
  ROUND(c.fob_promedio_capitulo, 2) AS fob_promedio_capitulo,
  ROUND(g.promedio_gral, 2)         AS promedio_gral,
  ROUND(c.fob_promedio_capitulo - g.promedio_gral, 2) AS diff,
  ROUND((c.fob_promedio_capitulo - g.promedio_gral) * 100 / g.promedio_gral, 2) AS pct_diff
FROM por_capitulo c
CROSS JOIN general g
ORDER BY pct_diff DESC;

-- Hallazgos:
-- 

### Ejercicio 6.3: Concentración de importadores

¿Cuánto del valor importado concentran los 20 mayores importadores? Mostrá nombre, FOB en USD, porcentaje del total, porcentaje acumulado y ranking.

*Pista: SUM(fob) OVER () para el total y SUM(fob) OVER (ORDER BY fob DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) para el acumulado.*

> Los nombres de empresas son públicos por ley, pero pensá bien qué mostrás en el portfolio.

In [0]:
%sql
WITH por_importador AS (
  SELECT nombre_importador, SUM(fob_dolar) AS fob
  FROM impo_items
  WHERE fob_dolar > 0
  GROUP BY nombre_importador
)
SELECT
  RANK() OVER (ORDER BY fob DESC) AS ranking,
  nombre_importador,
  ROUND(fob, 2) AS fob_usd,
  ROUND(fob * 100.0 / SUM(fob) OVER (), 2) AS pct_total,
  ROUND(SUM(fob) OVER (ORDER BY fob DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) * 100.0 / SUM(fob) OVER (), 2) AS pct_acumulado
FROM por_importador
ORDER BY ranking
LIMIT 20;

-- Hallazgos:
-- 

### Ejercicio 6.4: Top 3 importadores por capítulo químico (28 a 40)

Para cada capítulo NCM entre 28 y 40, mostrá los 3 importadores con mayor valor FOB.

*Pista: ROW_NUMBER() OVER (PARTITION BY capitulo_ncm ORDER BY fob DESC) y filtrar rn <= 3.*

In [0]:
%sql
WITH base AS (
  SELECT capitulo_ncm, nombre_importador, SUM(fob_dolar) AS fob
  FROM impo_items
  WHERE capitulo_ncm BETWEEN '28' AND '40'
    AND fob_dolar > 0
  GROUP BY capitulo_ncm, nombre_importador
),
ranking AS (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY capitulo_ncm ORDER BY fob DESC) AS rn
  FROM base
)
SELECT capitulo_ncm, rn AS puesto, nombre_importador, ROUND(fob, 2) AS fob_usd
FROM ranking
WHERE rn <= 3
ORDER BY capitulo_ncm, rn;

-- Hallazgos:
-- 

### Ejercicio 6.5: Análisis temporal (cuando haya más meses)

Con un solo corte (`202610`) no hay serie. Cuando cargues más meses, agrupá por `periodo` y mostrá: cantidad de ítems y FOB total, y calculá la variación contra el mes anterior.

*Pista: LAG(fob_total) OVER (ORDER BY periodo). `periodo` es texto AAAAMM, no una fecha completa.*

In [0]:
%sql
-- Cuando haya más de un mes cargado en Bronze

-- Hallazgos:
-- 

---
# PARTE 7: Documentación de hallazgos

### Ejercicio 7.1: Resumen ejecutivo
Documentá:
1. Total de registros analizados
2. Principales problemas de calidad encontrados
3. Porcentaje de datos válidos vs inválidos
4. Recomendaciones para la limpieza en Silver (reglas concretas: qué se descarta, qué se marca, qué se castea)

*Pista: combiná los resultados de los ejercicios anteriores.*

## Resumen ejecutivo (completar acá)

**1. Total de registros analizados:**
- 

**2. Principales problemas de calidad:**
- 

**3. % de datos válidos vs inválidos:**
- 

**4. Recomendaciones para Silver:**
- Grano: 
- Divisa distinta de DOL: 
- `fob_total` (total de destinación o de ítem): 
- Importador (truncado, sin CUIT): 
- Tablas de equivalencias necesarias (país, unidad de medida, concepto arancelario, NCM): 

In [0]:
%sql
-- (Opcional) Query de apoyo para el resumen

---
# ✅ Checklist de finalización
- [ ] Completé la exploración inicial (Parte 1)
- [ ] Analicé valores nulos y vacíos (Parte 2)
- [ ] Analicé distribuciones categóricas (Parte 3)
- [ ] Calculé estadísticas descriptivas (Parte 4)
- [ ] Identifiqué problemas de calidad (Parte 5)
- [ ] Apliqué Window Functions (Parte 6)
- [ ] Documenté mis hallazgos (Parte 7)
- [ ] Repetí el EDA con otro mes para validar las reglas de Silver
- [ ] Subí mi notebook a GitHub
- [ ] Creé un README explicando los hallazgos principales

**Tips:** el EDA es iterativo · documentá todo · usá CTEs · las Window Functions sirven para análisis comparativos · identificá valores placeholder (999, -1, 'N/A', 0).